In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O 

In [ ]:
train_df = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header = None)
train_label = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv' , header = None)
test_df = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv' , header =  None)

the first step in any classification problem is to check the distribution of each class

In [ ]:
train_label.head()

In [ ]:
train_label.value_counts() / len(train_label)

okay that is nice we don't have an unbalanced dataset

In [ ]:
train_df

as you can see we don't have any information about that competition let's give the feature usefull name

In [ ]:
columns_names = [f'feat{i}' for i in range(train_df.shape[1])]
new_names = {}
for i in range(train_df.shape[1]):
    new_names[train_df.columns[i]] = columns_names[i] 

train_df.rename(columns = new_names)

In [ ]:
train_df.info()

okay we don't have any missing data that is nice

In [ ]:
train_df.describe().T

to be honst that there is alot of feature it's hard to figure some insight but at least compare the mean and median for each feature ,let's  take a quick feel of the distribution of the data

In [ ]:
train_df.hist(bins = 30 , figsize = (12,10 ))
plt.subplots_adjust(
                    bottom=0.1   ,
                    top=1.2 ,
                    wspace=0.4,
                    hspace=0.4)
plt.show()

okay that seems that we have a normal distributed features which is the favourate distribution for machine learning models as i don't have feature information ican not EDA so let's prepare the pipeline

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score , RandomizedSearchCV , GridSearchCV
from scipy.stats import randint
from sklearn.mixture import GaussianMixture
from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.utils.validation import check_array , check_is_fitted

as all we have here is num feature so let's build a numerical pipeline

In [ ]:
num_preprocessing = make_pipeline( SimpleImputer(strategy = 'median') ,
                              StandardScaler() )

In [ ]:
class GaussianSimilarity(BaseEstimator , TransformerMixin):
    def __init__(self  , n_components = 3 ,covariance_type  ='full' ):
        self.n_components = n_components
        self.covariance_type = covariance_type
        
    def fit(self , X , y = None):
        self.gaussian_ = GaussianMixture(n_components =  self.n_components,
                                       covariance_type = self.covariance_type)
        self.gaussian_.fit(X)
        return self
    
    def transform(self , X):
        return self.gaussian_.predict_proba(X)
    
    def get_feature_names_out(self , names = None):
        return [f"Cluster {i} similarity" for i in range(self.n_components)]
    
cluster_simil = GaussianSimilarity(n_components = 3 ,covariance_type  ='full' )

In [ ]:
preprocessing = Pipeline([
    ('num_preprocessing' , num_preprocessing ), 
    ('mixure' , cluster_simil )
])

In [ ]:
full_pipeline = Pipeline([
    ('preprocessing' , preprocessing) ,
    ('random_forest' , RandomForestClassifier(random_state=42)),
])

In [ ]:
full_pipeline.get_feature_names_out

that is not the best way but let's keep things simple and fine-tuning our model

In [ ]:
param_dist = {
    'preprocessing__mixure__covariance_type' : ['spherical', 'tied', 'diag', 'full'],
    'preprocessing__mixure__n_components' :randint(low = 1 , high = 10),
    'random_forest__max_features' : randint(low = 1 , high = 10),
    'random_forest__min_samples_split' : randint(low = 2 , high= 10),
    'random_forest__min_samples_leaf' : randint(low = 1 , high = 10) , 
    'random_forest__n_estimators' : randint(low = 100 , high = 300) , 
}
rnd_search = RandomizedSearchCV(full_pipeline , param_distributions = param_dist , n_iter = 100 , cv = 10 ,
                               scoring = 'accuracy')

In [ ]:
rnd_search.fit(train_df , train_label.values.ravel())

In [ ]:
rnd_search.best_params_

In [ ]:
cv_results = pd.DataFrame(rnd_search.cv_results_)
cv_results.sort_values(by = 'mean_test_score' , ascending = False , inplace = True)

In [ ]:
cv_results

In [ ]:
final_model = rnd_search.best_estimator_

In [ ]:
prediction = final_model.predict(test_df)

In [ ]:
prediction_df = pd.DataFrame({'Id' : list(test_df.index+1) ,'Solution': prediction })

In [ ]:
prediction_df.to_csv('submission.csv', index = False)